# Milestone 3: Leakage-Free Feature Engineering

This notebook executes Steps 1, 2, and 3 of Milestone 3:
1. **Step 1: Split First** — Temporal separation into Train ($\le 2021$, $N=5,010$) and Test ($\ge 2022$, $N=2,525$).
2. **Step 2: Basic Features** — Production scope (`episodes`), platform (`original_network`), audience classification (`content_rating`), broadcast schedule (`aired_on`), and pitch genres (`genres`). Missingness is strictly isolated into an ablation group.
3. **Step 3: Attached Talent Track Records** — Expanding-window historical features (`director`, `screenwriter`, `cast_summary` `[Main Role]`) with Empirical Bayes smoothing ($m=3$), conservative date rules, fixed 5-year window variants, and deployment-style V1 vs. static V2 variants.

### Core Non-Leakage Principles:
> **1. Train-Only Fitting:** Every vocabulary, grouping, threshold, and global mean is fitted strictly on TRAIN and applied to TEST.
> **2. Strict 12-Month Rule:** Historical track records only aggregate titles premiering at least 365 days prior to the current drama's premiere date ($	ext{start\_date}_{	ext{prior}} \le 	ext{start\_date}_{	ext{current}} - 365	ext{ days}$).
> **3. Unknown vs. Debut:** "Unknown" credits map to `NaN` across all features (including $n_{	ext{prior}}$). Debuts (known talent, but 0 prior qualifying titles) receive $n_{	ext{prior}} = 0$ and reach = `NaN`.


In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Add src to path for reusable feature functions
sys.path.append(os.path.abspath('..'))
from src.features import (
    load_labeled_data,
    temporal_train_test_split,
    fit_basic_features,
    transform_basic_features,
    prepare_talent_columns,
    build_talent_inverted_index,
    compute_talent_priors
)

# Load scoped & labeled dataset from Step 2E
df = load_labeled_data('../data/processed/dramas_labeled.csv')
print(f"Loaded labeled dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")


Loaded labeled dataset: 7,535 rows, 35 columns


## Step 1: Split First (Temporal Train / Test Split)

- **Train Cohort:** Dramas released in $2015 - 2021$ (`release_year <= 2021`).
- **Test Cohort:** Dramas released in $2022 - 2025$ (`release_year >= 2022`).


In [2]:
train_df, test_df = temporal_train_test_split(df, cutoff_year=2021)

print(f"Train set: {len(train_df):,} dramas ({len(train_df)/len(df)*100:.1f}%) | Years: {train_df['release_year'].min()} - {train_df['release_year'].max()}")
print(f"Test set:  {len(test_df):,} dramas ({len(test_df)/len(df)*100:.1f}%) | Years: {test_df['release_year'].min()} - {test_df['release_year'].max()}")

# Automated boundary assertions
assert train_df['release_year'].max() <= 2021, "Data leakage: Train contains titles released after 2021!"
assert test_df['release_year'].min() >= 2022, "Data leakage: Test contains titles released before 2022!"
assert len(train_df) + len(test_df) == len(df), "Row loss detected during train/test split!"
print("\n[PASSED] Temporal isolation assertions verified.")


Train set: 5,010 dramas (66.5%) | Years: 2015 - 2021
Test set:  2,525 dramas (33.5%) | Years: 2022 - 2025

[PASSED] Temporal isolation assertions verified.


In [3]:
# Country breakdown in train vs test
split_country = pd.DataFrame({
    'Train Count': train_df['country'].value_counts(),
    'Train %': (train_df['country'].value_counts(normalize=True)*100).round(1),
    'Test Count': test_df['country'].value_counts(),
    'Test %': (test_df['country'].value_counts(normalize=True)*100).round(1)
})
print("=== COUNTRY BREAKDOWN ACROSS SPLIT ===")
print(split_country.to_string())

# Tier mix in train vs test
split_tiers = pd.DataFrame({
    'Train Tier %': (train_df['target_tier'].value_counts(normalize=True)[['High', 'Medium', 'Low']]*100).round(1),
    'Test Tier %': (test_df['target_tier'].value_counts(normalize=True)[['High', 'Medium', 'Low']]*100).round(1)
})
print("\n=== TARGET TIER BALANCE (%) ===")
print(split_tiers.to_string())


=== COUNTRY BREAKDOWN ACROSS SPLIT ===
             Train Count  Train %  Test Count  Test %
country                                              
China               1926     38.4         532    21.1
Japan                802     16.0         452    17.9
South Korea         1163     23.2         751    29.7
Thailand            1119     22.3         790    31.3

=== TARGET TIER BALANCE (%) ===
             Train Tier %  Test Tier %
target_tier                           
High                 25.3         25.3
Medium               50.0         49.9
Low                  24.7         24.8


## Step 2: Basic Features (Core vs. Missingness Groups)

- **Core Group ($N=41$):** `episodes`, `country` (single categorical string), 7 weekday indicators, `original_network`, `content_rating`, 30 multi-hot pitch genres. Missingness is strictly represented as `NaN`.
- **Missingness Group ($N=4$):** Ablation flags capturing page-completeness signals (`aired_on_is_missing`, `content_rating_is_missing`, `network_is_missing`, `genre_is_missing`).


In [4]:
# 1. Fit parameters on train only
params = fit_basic_features(train_df)

# 2. Transform train and test
train_core, train_miss = transform_basic_features(train_df, params)
test_core, test_miss = transform_basic_features(test_df, params)

print("=== FEATURE TRANSFORMATION SUMMARY ===")
print(f"Train Core shape:        {train_core.shape}")
print(f"Test Core shape:         {test_core.shape}")
print(f"Train Missingness shape: {train_miss.shape}")
print(f"Test Missingness shape:  {test_miss.shape}")

# Assert identical columns
assert train_core.columns.tolist() == test_core.columns.tolist(), "Core column mismatch between train and test!"
assert train_miss.columns.tolist() == test_miss.columns.tolist(), "Missingness column mismatch between train and test!"

# Validation: No missingness encoding in Core except NaN
day_cols = [c for c in train_core.columns if c.startswith('aired_on_')]
genre_cols = [c for c in train_core.columns if c.startswith('genre_')]

assert train_core.loc[train_df['aired_on'] == 'Unknown', day_cols].isna().all().all()
assert train_core.loc[train_df['aired_on'] != 'Unknown', day_cols].notna().all().all()
assert train_core.loc[train_df['genres'] == 'Unknown', genre_cols].isna().all().all()
assert train_core.loc[train_df['genres'] != 'Unknown', genre_cols].notna().all().all()
assert train_core.loc[train_df['original_network'] == 'Unknown', 'original_network'].isna().all()
assert train_core.loc[train_df['content_rating'] == 'Not Yet Rated', 'content_rating'].isna().all()

print("\n[PASSED] Verification confirmed: No core column combination encodes missingness except NaN.")


=== FEATURE TRANSFORMATION SUMMARY ===
Train Core shape:        (5010, 41)
Test Core shape:         (2525, 41)
Train Missingness shape: (5010, 4)
Test Missingness shape:  (2525, 4)

[PASSED] Verification confirmed: No core column combination encodes missingness except NaN.


## Step 3: Attached Talent Track Records (Expanding-Window Priors)

### Rules & Design:
1. **Name Normalization:** Lowercase, collapse whitespace, strip parentheses `(...)`, remove accents. No fuzzy matching.
2. **Scope of Talent:**
   - `director`: All listed directors (comma-delimited).
   - `screenwriter`: All listed screenwriters (comma-delimited).
   - `cast_summary`: Lead actors only (tokens containing `[Main Role]`). Dramas with no `[Main Role]` receive `NaN`.
3. **Unknown vs. Debut:**
   - If a credit field is `Unknown` / missing: all features for that role are `NaN` (including $n_{	ext{prior}}$).
   - If known, but no qualifying prior titles exist: $n_{	ext{prior}} = 0$ and reach = `NaN`.
4. **Dates & 12-Month Rule:**
   - A prior drama qualifies if: $	ext{eff\_pri\_date} \le 	ext{eff\_cur\_date} - 365	ext{ days}$ and $	ext{id}_{	ext{prior}} 
e 	ext{id}_{	ext{current}}$.
   - Missing `start_date` rule: Jan 1 of `release_year` when evaluated as current drama; Dec 31 of `release_year` when evaluated as a prior candidate.
5. **Shrinkage Smoothing:**
   $$\hat{\mu}_{	ext{smoothed}} = rac{n_{	ext{prior}} \cdot ar{y} + m \cdot \mu_{	ext{global}}}{n_{	ext{prior}} + m}$$
   where $\mu_{	ext{global}} = 0.5022$ (computed strictly on Train).


In [5]:
import unicodedata
import re

# Inspect name variations differing only by a leading short token (potential nickname / surname)
prep_train = prepare_talent_columns(train_df)
names_by_country = {'China': set(), 'Japan': set(), 'South Korea': set(), 'Thailand': set()}

for idx, row in prep_train.iterrows():
    c = row['country']
    for role_list in [row['clean_dirs'], row['clean_scrs'], row['clean_cast']]:
        if role_list:
            for name in role_list:
                names_by_country[c].add(name)

print("=== TALENT NAME NORMALIZATION & NICKNAME DIAGNOSTIC ===")
for c, nset in names_by_country.items():
    pairs = []
    for name in nset:
        parts = name.split()
        if len(parts) >= 2 and len(parts[0]) <= 5:
            base = ' '.join(parts[1:])
            if base in nset:
                pairs.append((parts[0], base, name))
    print(f"{c:12s}: {len(nset):4d} unique names | {len(pairs):3d} pairs differing by leading short token")

print("\nFinding: Thailand has 0 pairs differing by nickname (MDL consistently standardizes Thai display names).")
print("China (123) and Korea (83) reflect mononyms vs full names. Strict exact matching is maintained without fuzzy conflation.")


=== TALENT NAME NORMALIZATION & NICKNAME DIAGNOSTIC ===
China       : 3302 unique names | 123 pairs differing by leading short token
Japan       : 1707 unique names |   9 pairs differing by leading short token
South Korea : 2570 unique names |  83 pairs differing by leading short token
Thailand    : 1684 unique names |   0 pairs differing by leading short token

Finding: Thailand has 0 pairs differing by nickname (MDL consistently standardizes Thai display names).
China (123) and Korea (83) reflect mononyms vs full names. Strict exact matching is maintained without fuzzy conflation.


### 3.1 Empirical Bayes Smoothing Sensitivity Check ($m \in \{1, 3, 10\}$)
We evaluate how the shrinkage parameter $m$ affects both predictive correlation and Root Mean Squared Error (RMSE) against `target_pct` on $2020-2021$ train dramas ($N=1,490$) using only earlier train dramas as priors.


In [6]:
global_mean = train_df['target_pct'].mean()
print(f"Train Global Mean target_pct: {global_mean:.4f}")

# Train-only inverted index
train_prepared = prepare_talent_columns(train_df)
train_dir_idx, train_scr_idx, train_cast_idx = build_talent_inverted_index(train_prepared)

val_20_21 = train_prepared[train_prepared['release_year'].isin([2020, 2021])].copy()

print("\n=== SMOOTHING SENSITIVITY: CORRELATION, RMSE & SAMPLE SIZES (2020-2021 TRAIN) ===")
for m_val in [1, 3, 10]:
    p_val = compute_talent_priors(val_20_21, train_dir_idx, train_scr_idx, train_cast_idx, m=m_val, global_mean=global_mean)
    
    # Director
    val_d = val_20_21['target_pct'].loc[p_val['director_prior_reach_max'].notna()]
    pred_d = p_val['director_prior_reach_max'].dropna()
    r_d = pred_d.corr(val_d)
    rmse_d = np.sqrt(np.mean((pred_d - val_d)**2))
    
    # Screenwriter
    val_s = val_20_21['target_pct'].loc[p_val['screenwriter_prior_reach_max'].notna()]
    pred_s = p_val['screenwriter_prior_reach_max'].dropna()
    r_s = pred_s.corr(val_s)
    rmse_s = np.sqrt(np.mean((pred_s - val_s)**2))
    
    # Cast
    val_c = val_20_21['target_pct'].loc[p_val['cast_prior_reach_max'].notna()]
    pred_c = p_val['cast_prior_reach_max'].dropna()
    r_c = pred_c.corr(val_c)
    rmse_c = np.sqrt(np.mean((pred_c - val_c)**2))
    
    print(f"m = {m_val:2d}:")
    print(f"  Director:     r = {r_d:+.4f} | RMSE = {rmse_d:.4f} | N = {len(pred_d):,}")
    print(f"  Screenwriter: r = {r_s:+.4f} | RMSE = {rmse_s:.4f} | N = {len(pred_s):,}")
    print(f"  Lead Cast:    r = {r_c:+.4f} | RMSE = {rmse_c:.4f} | N = {len(pred_c):,}")

print("\nMethodological Choice Note:")
print("m = 1 scored slightly higher on correlation (+0.6084 vs +0.5841 for cast) and lower RMSE (0.2238 vs 0.2267).")
print("However, m = 3 was selected as a conservative hyperparameter choice to maintain stability against extreme singletons")
print("(given that 52.4% of lead actors in train appear only once).")


Train Global Mean target_pct: 0.5022



=== SMOOTHING SENSITIVITY: CORRELATION, RMSE & SAMPLE SIZES (2020-2021 TRAIN) ===
m =  1:
  Director:     r = +0.4268 | RMSE = 0.2363 | N = 561
  Screenwriter: r = +0.4590 | RMSE = 0.2466 | N = 488
  Lead Cast:    r = +0.6084 | RMSE = 0.2238 | N = 925


m =  3:
  Director:     r = +0.4245 | RMSE = 0.2405 | N = 561
  Screenwriter: r = +0.4497 | RMSE = 0.2545 | N = 488
  Lead Cast:    r = +0.5841 | RMSE = 0.2267 | N = 925
m = 10:
  Director:     r = +0.4075 | RMSE = 0.2592 | N = 561
  Screenwriter: r = +0.4218 | RMSE = 0.2735 | N = 488
  Lead Cast:    r = +0.5372 | RMSE = 0.2464 | N = 925

Methodological Choice Note:
m = 1 scored slightly higher on correlation (+0.6084 vs +0.5841 for cast) and lower RMSE (0.2238 vs 0.2267).
However, m = 3 was selected as a conservative hyperparameter choice to maintain stability against extreme singletons
(given that 52.4% of lead actors in train appear only once).


### 3.2 Building V1 and V2 Talent Prior Features
- **Train Priors:** Computed from earlier train dramas ($\le 2021$) in both variants.
- **Test Priors V1 (Deployment-Style Evaluation):** Priors computed from Train + earlier Test dramas ($\le 	ext{start\_date} - 365	ext{ days}$).
- **Test Priors V2 (Static Train-Only Benchmark):** Priors computed strictly from Train dramas ($\le 2021$).


In [7]:
m_chosen = 3.0

# 1. Prepare datasets
all_prepared = prepare_talent_columns(df)
test_prepared = prepare_talent_columns(test_df)

# Inverted indices
all_dir_idx, all_scr_idx, all_cast_idx = build_talent_inverted_index(all_prepared)

# 2. Compute priors
print("Computing Train Priors...")
train_talent_priors = compute_talent_priors(train_prepared, train_dir_idx, train_scr_idx, train_cast_idx, m=m_chosen, global_mean=global_mean)

print("Computing Test Priors (Variant 1: Deployment-Style Train + Test)...")
test_talent_v1 = compute_talent_priors(test_prepared, all_dir_idx, all_scr_idx, all_cast_idx, m=m_chosen, global_mean=global_mean)

print("Computing Test Priors (Variant 2: Static Train Only)...")
test_talent_v2 = compute_talent_priors(test_prepared, train_dir_idx, train_scr_idx, train_cast_idx, m=m_chosen, global_mean=global_mean)

print(f"\nTrain Talent shape:   {train_talent_priors.shape}")
print(f"Test Talent V1 shape: {test_talent_v1.shape}")
print(f"Test Talent V2 shape: {test_talent_v2.shape}")


Computing Train Priors...


Computing Test Priors (Variant 1: Deployment-Style Train + Test)...


Computing Test Priors (Variant 2: Static Train Only)...



Train Talent shape:   (5010, 14)
Test Talent V1 shape: (2525, 14)
Test Talent V2 shape: (2525, 14)


### 3.3 Historical Experience ($n_{	ext{prior}}$) vs. Time Analysis
We inspect how prior title counts behave across release years in the training set.
Because the scraped dataset starts in 2015, all-time expanding counts are left-censored: earlier titles mechanically have shorter historical look-backs.
We evaluate an alternative: a **5-year fixed look-back window** ($	ext{cutoff} - 5	ext{ years} < 	ext{prior} \le 	ext{cutoff}$).


In [8]:
# Compare All-Time vs 5-Year Fixed Lookback
n_prior_eval = train_talent_priors[['director_n_prior_max', 'director_n_prior_5yr_max',
                                   'screenwriter_n_prior_max', 'screenwriter_n_prior_5yr_max',
                                   'cast_n_prior_max', 'cast_n_prior_5yr_max',
                                   'cast_n_prior_mean', 'cast_n_prior_5yr_mean']].copy()
n_prior_eval['release_year'] = train_df['release_year']

print("=== MEAN N_PRIOR BY RELEASE YEAR: ALL-TIME vs. 5-YEAR FIXED WINDOW ===")
yearly_means = n_prior_eval.groupby('release_year').mean().round(2)
print(yearly_means.to_string())

print("\n=== CORRELATION OF N_PRIOR WITH RELEASE_YEAR ===")
for col in n_prior_eval.columns:
    if col != 'release_year':
        r = n_prior_eval[col].corr(n_prior_eval['release_year'])
        n = n_prior_eval[col].notna().sum()
        print(f"{col:28s}: r with release_year = {r:+.4f} (N={n:,})")

print("\nKey Finding:")
print("All-time n_prior features correlate +0.42 with release_year because the corpus starts in 2015 (left censoring).")
print("A 5-year fixed window standardizes history length once 5 years of data are available (e.g. from 2021 onwards and across Test).")


=== MEAN N_PRIOR BY RELEASE YEAR: ALL-TIME vs. 5-YEAR FIXED WINDOW ===
              director_n_prior_max  director_n_prior_5yr_max  screenwriter_n_prior_max  screenwriter_n_prior_5yr_max  cast_n_prior_max  cast_n_prior_5yr_max  cast_n_prior_mean  cast_n_prior_5yr_mean
release_year                                                                                                                                                                                          
2015                          0.00                      0.00                      0.00                          0.00              0.00                  0.00               0.00                   0.00
2016                          0.47                      0.47                      0.40                          0.40              0.65                  0.65               0.30                   0.30
2017                          1.24                      1.24                      1.15                          1.15              1.5

### 3.4 Leakage Checks & Validation Tests
1. **Automated Assertion:** Enforced within `compute_talent_priors`: every prior has $	ext{eff\_pri\_date} \le 	ext{eff\_cur\_date} - 365	ext{ days}$ and $	ext{id}_{	ext{prior}} 
e 	ext{id}_{	ext{current}}$.
2. **3-Drama Trace:** Veteran (*Fai Gam Prae*, 2020), Debut (*Dok Mai Tai Mek*, 2015), Unknown credits (*So Much in Love*, 2020).
3. **Target Permutation Shuffle Test:** Randomly shuffle `target_pct` in Train and recompute talent features; correlation must collapse to near zero.


In [9]:
def trace_drama(row_idx, df_source):
    row = df_source.loc[row_idx]
    cur_date = row['eff_cur_date']
    cutoff = cur_date - pd.Timedelta(days=365)
    print(f"=== TRACE: {row['name']} ===")
    print(f"Country: {row['country']} | Release: {row['release_year']} | Premiere: {cur_date.strftime('%Y-%m-%d')} | Cutoff: <= {cutoff.strftime('%Y-%m-%d')}")
    
    for role, col, r_idx in [('Director', 'clean_dirs', all_dir_idx), ('Screenwriter', 'clean_scrs', all_scr_idx), ('Lead Cast', 'clean_cast', all_cast_idx)]:
        persons = row[col]
        if persons is None:
            print(f"  [{role}]: Field is Unknown -> ALL features = NaN")
        else:
            print(f"  [{role}]: Listed: {persons}")
            for p in persons:
                priors = [x for x in r_idx.get(p, []) if x[0] <= cutoff and x[2] != row_idx]
                print(f"    - '{p}': {len(priors)} qualifying priors")
                for pr in priors[:2]:
                    print(f"        * Prior: '{pr[3]}' (Air: {pr[0].strftime('%Y-%m-%d')}, Reach: {pr[1]:.4f})")
    print("=" * 60)

# 1. Veteran drama: Fai Gam Prae (2020)
trace_drama(3, all_prepared)

# 2. Pure Debut drama: Dok Mai Tai Mek (2015)
trace_drama(754, all_prepared)

# 3. Unknown credits drama: So Much in Love (2020)
trace_drama(89, all_prepared)


=== TRACE: Fai Gam Prae (2020) ===
Country: Thailand | Release: 2020 | Premiere: 2020-09-23 | Cutoff: <= 2019-09-24
  [Director]: Listed: ['koo ekkasit trakulkasemsuk']
    - 'koo ekkasit trakulkasemsuk': 20 qualifying priors
        * Prior: 'Boy for Rent (2019)' (Air: 2019-05-10, Reach: 0.9494)
        * Prior: 'The Fallen Leaf (2019)' (Air: 2019-06-11, Reach: 0.8876)
  [Screenwriter]: Listed: ['thunyaluck julapong']
    - 'thunyaluck julapong': 11 qualifying priors
        * Prior: 'Ngao (2018)' (Air: 2018-11-05, Reach: 0.5116)
        * Prior: 'Club Friday Season 11 (2019)' (Air: 2019-03-02, Reach: 0.1910)
  [Lead Cast]: Listed: ['kaew jarinya sirimongkolsakul', 'marina sadanun balenciaga', 'new chaiyapol poupart', 'iang sittha sapanuchart']
    - 'kaew jarinya sirimongkolsakul': 8 qualifying priors
        * Prior: 'Rak Chan Sawan Jat Hai (2018)' (Air: 2018-07-14, Reach: 0.1221)
        * Prior: 'Pak (2018)' (Air: 2018-09-03, Reach: 0.2500)
    - 'marina sadanun balenciaga': 2 qua

In [10]:
print("=== SHUFFLE TEST (PERMUTE TARGET_PCT IN TRAIN) ===")
np.random.seed(42)
shuffled_train = train_prepared.copy()
shuffled_train['target_pct'] = np.random.permutation(train_prepared['target_pct'].values)

shuf_dir, shuf_scr, shuf_cast = build_talent_inverted_index(shuffled_train)
shuf_priors = compute_talent_priors(shuffled_train, shuf_dir, shuf_scr, shuf_cast, m=m_chosen, global_mean=global_mean)

shuffle_report = []
for col in train_talent_priors.columns:
    if 'reach' in col:
        r_true = train_talent_priors[col].corr(train_prepared['target_pct'])
        r_shuf = shuf_priors[col].corr(shuffled_train['target_pct'])
        shuffle_report.append({'Feature': col, 'True Correlation': round(r_true, 4), 'Shuffled Correlation': round(r_shuf, 4)})

print(pd.DataFrame(shuffle_report).to_string(index=False))
print("\n[PASSED] Shuffle test confirms zero structural leakage: correlations collapse to ~0 when target is permuted.")


=== SHUFFLE TEST (PERMUTE TARGET_PCT IN TRAIN) ===


                      Feature  True Correlation  Shuffled Correlation
     director_prior_reach_max            0.4250               -0.0311
    director_prior_reach_mean            0.4333               -0.0179
 screenwriter_prior_reach_max            0.4375                0.0322
screenwriter_prior_reach_mean            0.4691                0.0282
         cast_prior_reach_max            0.5703                0.0191
        cast_prior_reach_mean            0.5992                0.0236

[PASSED] Shuffle test confirms zero structural leakage: correlations collapse to ~0 when target is permuted.


### 3.5 Feature Diagnostics: Correlations & Missingness Breakdown
We report the correlation of each talent feature with `target_pct` alongside the number of non-NaN rows $N$, and analyze missingness rates.


In [11]:
talent_cols = [
    'director_n_prior_max', 'director_n_prior_5yr_max', 'director_prior_reach_max', 'director_prior_reach_mean',
    'screenwriter_n_prior_max', 'screenwriter_n_prior_5yr_max', 'screenwriter_prior_reach_max', 'screenwriter_prior_reach_mean',
    'cast_n_prior_max', 'cast_n_prior_5yr_max', 'cast_n_prior_mean', 'cast_n_prior_5yr_mean',
    'cast_prior_reach_max', 'cast_prior_reach_mean'
]

rows = []
for col in talent_cols:
    valid_tr = pd.DataFrame({'f': train_talent_priors[col], 'y': train_df['target_pct']}).dropna()
    r_tr = valid_tr['f'].corr(valid_tr['y'])
    n_tr = len(valid_tr)
    
    valid_v1 = pd.DataFrame({'f': test_talent_v1[col], 'y': test_df['target_pct']}).dropna()
    r_v1 = valid_v1['f'].corr(valid_v1['y'])
    n_v1 = len(valid_v1)
    
    valid_v2 = pd.DataFrame({'f': test_talent_v2[col], 'y': test_df['target_pct']}).dropna()
    r_v2 = valid_v2['f'].corr(valid_v2['y'])
    n_v2 = len(valid_v2)
    
    rows.append({
        'Feature': col,
        'Train Corr (N)': f"{r_tr:+.4f} (N={n_tr:,})",
        'Test V1 Corr (N)': f"{r_v1:+.4f} (N={n_v1:,})",
        'Test V2 Corr (N)': f"{r_v2:+.4f} (N={n_v2:,})"
    })

print("=== TALENT FEATURE CORRELATIONS WITH TARGET REACH (WITH SAMPLE SIZES) ===")
print(pd.DataFrame(rows).to_string(index=False))

# NaN share table
nan_table = pd.DataFrame({
    'Train NaN %': (train_talent_priors.isna().mean() * 100).round(1),
    'Test V1 NaN %': (test_talent_v1.isna().mean() * 100).round(1),
    'Test V2 NaN %': (test_talent_v2.isna().mean() * 100).round(1),
    'Train CompleteInfo NaN %': (train_talent_priors[train_df['complete_info'] == 1].isna().mean() * 100).round(1),
    'Train Incomplete NaN %': (train_talent_priors[train_df['complete_info'] == 0].isna().mean() * 100).round(1)
})
print("\n=== SHARE OF NaN PER TALENT FEATURE (%) ===")
print(nan_table.to_string())

# Reach features NaN rate by release_year and country (Train vs. Test V1)
reach_cols = [
    'director_prior_reach_max', 'director_prior_reach_mean',
    'screenwriter_prior_reach_max', 'screenwriter_prior_reach_mean',
    'cast_prior_reach_max', 'cast_prior_reach_mean'
]
train_reach = pd.concat([train_df[['release_year', 'country']].assign(Split='Train'), train_talent_priors[reach_cols]], axis=1)
test_reach = pd.concat([test_df[['release_year', 'country']].assign(Split='Test (V1)'), test_talent_v1[reach_cols]], axis=1)
full_reach = pd.concat([train_reach, test_reach], ignore_index=True)

print("\n=== SHARE OF NaN FOR REACH FEATURES BY RELEASE YEAR (Train vs. Test V1) (%) ===")
print(full_reach.groupby(['Split', 'release_year'])[reach_cols].apply(lambda g: (g.isna().mean() * 100).round(1)).to_string())

print("\n=== SHARE OF NaN FOR REACH FEATURES BY COUNTRY (Train vs. Test V1) (%) ===")
print(full_reach.groupby(['Split', 'country'])[reach_cols].apply(lambda g: (g.isna().mean() * 100).round(1)).to_string())


=== TALENT FEATURE CORRELATIONS WITH TARGET REACH (WITH SAMPLE SIZES) ===
                      Feature    Train Corr (N)  Test V1 Corr (N)  Test V2 Corr (N)
         director_n_prior_max +0.0718 (N=2,934) -0.0474 (N=1,522) -0.0692 (N=1,522)
     director_n_prior_5yr_max +0.0684 (N=2,934) -0.0393 (N=1,522) -0.0681 (N=1,522)
     director_prior_reach_max +0.4250 (N=1,594) +0.4945 (N=1,173) +0.4595 (N=1,122)
    director_prior_reach_mean +0.4333 (N=1,594) +0.4933 (N=1,173) +0.4586 (N=1,122)
     screenwriter_n_prior_max -0.0549 (N=2,529) -0.1016 (N=1,298) -0.1452 (N=1,298)
 screenwriter_n_prior_5yr_max -0.0537 (N=2,529) -0.0646 (N=1,298) -0.1203 (N=1,298)
 screenwriter_prior_reach_max +0.4375 (N=1,256) +0.4145 (N=1,010)   +0.3843 (N=979)
screenwriter_prior_reach_mean +0.4691 (N=1,256) +0.4418 (N=1,010)   +0.4131 (N=979)
             cast_n_prior_max +0.1277 (N=4,704) +0.0219 (N=2,324) +0.0136 (N=2,324)
         cast_n_prior_5yr_max +0.1293 (N=4,704) +0.0437 (N=2,324) +0.0271 (N=2,324)
  

## Step 4: Synopsis Text Cleaning & Simple Feature Extraction

In this step, we prepare synopsis text for both predictive modeling and downstream retrieval:
1. **Boilerplate and Leakage Removal:**
   - Stripping `(YYYY)` release year patterns in parentheses.
   - Removing `Source:` attribution lines (e.g., `(Source: ...)`, `Source: Baidu`, etc.).
   - Removing MDL `~~` metadata note lines (e.g., `~~ Adapted from...`, `~~ Remake of...`).
   - Removing `Edit Translation` and translation credit markers.
   - Normalizing extra whitespace.
2. **Text Feature (`synopsis_length`):**
   - Count of words in `synopsis_clean`.
   - Set strictly to `NaN` when missing/empty (NOT 0, ensuring missingness is not conflated with zero length).
   - Assigned to a dedicated switchable `"text"` feature group (**OFF by default** due to page-completeness bias).
3. **Retrieval Corpus Export (`synopses_clean.csv`):**
   - Cleaned synopses saved for all labeled dramas to `data/processed/synopses_clean.csv` (gitignored), keyed by `url` and `name`.
   - Dramas with empty synopses ($N=1,038$) will be excluded from the Milestone 6 vector index.


In [12]:
from src.features import clean_synopsis_text, compute_synopsis_features

syn_series = df['synopsis'].dropna().astype(str)

# 1. 5 Most common boilerplate patterns
bp_patterns = [
    ("Tilde note lines ('~~ ...')", r'(?m)^\s*~~.*$'),
    ("Source attribution ('Source: ...')", r'(?i)[(\[]?\s*source\s*:.*?(?:[)\]]|\n|$)'),
    ("Year in parentheses '(YYYY)'", r'\(\s*(?:19|20)\d{2}\s*\)'),
    ("Translation notes ('Translated by', 'Translation =')", r'(?i)\(?\s*translat(?:ed\s+by|ion\s*=).*?(?:\)|\n|$)'),
    ("Edit Translation marker", r'(?i)edit\s+translation'),
]

print("=== 5 MOST COMMON BOILERPLATE PATTERNS ===")
for name, pat in bp_patterns:
    c = syn_series.apply(lambda s: bool(re.search(pat, s))).sum()
    print(f"- {name:50s}: {c:5d} dramas ({c/len(df)*100:4.1f}% of total, {c/len(syn_series)*100:4.1f}% of non-null)")

# 2. Year mentions in synopses
c_any_yr = syn_series.apply(lambda s: bool(re.search(r'\b(?:19\d{2}|20\d{2})\b', s))).sum()
c_rel_yr = sum(str(int(r['release_year'])) in r['synopsis'] for _, r in df.dropna(subset=['synopsis']).iterrows())
c_bare_before = sum(
    bool(re.search(r'\b' + str(int(r['release_year'])) + r'\b', clean_synopsis_text(r['synopsis'])))
    for _, r in df.dropna(subset=['synopsis']).iterrows()
)
c_bare_after = sum(
    bool(re.search(r'\b' + str(int(r['release_year'])) + r'\b', clean_synopsis_text(r['synopsis'], r['release_year'])))
    for _, r in df.dropna(subset=['synopsis']).iterrows()
)

print("\n=== YEAR MENTIONS IN SYNOPSES ===")
print(f"- Mentions any 4-digit year (1900-2099): {c_any_yr:4d} dramas ({c_any_yr/len(syn_series)*100:.1f}%)")
print(f"- Mentions drama's exact release_year:   {c_rel_yr:4d} dramas ({c_rel_yr/len(syn_series)*100:.1f}%)")
print(f"- Bare mention of own release_year in synopsis_clean: {c_bare_before:2d} dramas (stripped -> {c_bare_after} remaining)")

# 3. Sequel / Season mentions
c_season = syn_series.apply(lambda s: bool(re.search(r'(?i)\bseason\b', s))).sum()
c_sequel = syn_series.apply(lambda s: bool(re.search(r'(?i)\bsequel\b', s))).sum()
c_phrases = syn_series.apply(lambda s: bool(re.search(r'(?i)\b(?:sequel|prequel|season\s*\d+|2nd\s*season|second\s*season)\b', s))).sum()

print("\n=== SEQUEL / SEASON MENTIONS ===")
print(f"- Mentions 'season' (any form): {c_season:3d} dramas ({c_season/len(syn_series)*100:.1f}%)")
print(f"- Mentions 'sequel' (any form): {c_sequel:3d} dramas ({c_sequel/len(syn_series)*100:.1f}%)")
print(f"- Mentions sequel/season phrases (sequel, prequel, season N, etc.): {c_phrases:3d} dramas ({c_phrases/len(syn_series)*100:.1f}%)")


=== 5 MOST COMMON BOILERPLATE PATTERNS ===
- Tilde note lines ('~~ ...')                       :  1631 dramas (21.6% of total, 24.8% of non-null)


- Source attribution ('Source: ...')                :    41 dramas ( 0.5% of total,  0.6% of non-null)
- Year in parentheses '(YYYY)'                      :     7 dramas ( 0.1% of total,  0.1% of non-null)


- Translation notes ('Translated by', 'Translation ='):     5 dramas ( 0.1% of total,  0.1% of non-null)


- Edit Translation marker                           :     1 dramas ( 0.0% of total,  0.0% of non-null)



=== YEAR MENTIONS IN SYNOPSES ===
- Mentions any 4-digit year (1900-2099):  334 dramas (5.1%)
- Mentions drama's exact release_year:     60 dramas (0.9%)
- Bare mention of own release_year in synopsis_clean: 31 dramas (stripped -> 0 remaining)



=== SEQUEL / SEASON MENTIONS ===
- Mentions 'season' (any form):  60 dramas (0.9%)
- Mentions 'sequel' (any form):  13 dramas (0.2%)
- Mentions sequel/season phrases (sequel, prequel, season N, etc.):  32 dramas (0.5%)


In [13]:
# Compute synopsis features
train_text = compute_synopsis_features(train_df)
test_text = compute_synopsis_features(test_df)

print("=== SYNOPSIS FEATURE SUMMARY ===")
print(f"Train empty/missing synopsis: {train_text['synopsis_length'].isna().sum():4d} ({train_text['synopsis_length'].isna().mean()*100:.1f}%)")
print(f"Test empty/missing synopsis:  {test_text['synopsis_length'].isna().sum():4d} ({test_text['synopsis_length'].isna().mean()*100:.1f}%)")

tr_len = train_text['synopsis_length'].dropna()
print(f"\nTrain word count percentiles (25, 50, 75, 90, 99): {np.percentile(tr_len, [25, 50, 75, 90, 99])}")
print(f"Train word count min: {tr_len.min()}, max: {tr_len.max()}, median: {tr_len.median()}, mean: {tr_len.mean():.1f}")

corr_watchers_tr = train_df['log_watchers'].corr(train_text['synopsis_length'])
corr_target_tr = train_df['target_pct'].corr(train_text['synopsis_length'])
corr_watchers_te = test_df['log_watchers'].corr(test_text['synopsis_length'])
corr_target_te = test_df['target_pct'].corr(test_text['synopsis_length'])

print(f"\nCorrelation of synopsis_length with log_watchers (Train, N={len(tr_len)}): {corr_watchers_tr:+.4f}")
print(f"Correlation of synopsis_length with target_pct   (Train, N={len(tr_len)}): {corr_target_tr:+.4f}")
print(f"Correlation of synopsis_length with log_watchers (Test,  N={test_text['synopsis_length'].notna().sum()}): {corr_watchers_te:+.4f}")
print(f"Correlation of synopsis_length with target_pct   (Test,  N={test_text['synopsis_length'].notna().sum()}): {corr_target_te:+.4f}")


=== SYNOPSIS FEATURE SUMMARY ===
Train empty/missing synopsis:  765 (15.3%)
Test empty/missing synopsis:   273 (10.8%)

Train word count percentiles (25, 50, 75, 90, 99): [ 47.    79.   116.   159.   260.56]
Train word count min: 1.0, max: 431.0, median: 79.0, mean: 88.1

Correlation of synopsis_length with log_watchers (Train, N=4245): +0.1183
Correlation of synopsis_length with target_pct   (Train, N=4245): +0.1775
Correlation of synopsis_length with log_watchers (Test,  N=2252): +0.1952
Correlation of synopsis_length with target_pct   (Test,  N=2252): +0.2081


In [14]:
# Export clean synopses for all labeled dramas (with bare release_year stripped)
synopses_export = pd.DataFrame({
    'url': df['url'],
    'name': df['name'],
    'release_year': df['release_year'],
    'country': df['country'],
    'synopsis_clean': [clean_synopsis_text(r['synopsis'], r['release_year']) for _, r in df.iterrows()]
})
synopses_export.to_csv('../data/processed/synopses_clean.csv', index=False)

n_empty = (synopses_export['synopsis_clean'] == '').sum()
n_valid = (synopses_export['synopsis_clean'] != '').sum()
print("=== RETRIEVAL CORPUS EXPORT ===")
print(f"Total dramas saved:   {len(synopses_export)} -> data/processed/synopses_clean.csv")
print(f"Valid clean synopses: {n_valid} ({n_valid/len(synopses_export)*100:.1f}%)")
print(f"Empty synopses (to be excluded from retrieval index): {n_empty} ({n_empty/len(synopses_export)*100:.1f}%)")


=== RETRIEVAL CORPUS EXPORT ===
Total dramas saved:   7535 -> data/processed/synopses_clean.csv
Valid clean synopses: 6497 (86.2%)
Empty synopses (to be excluded from retrieval index): 1038 (13.8%)


### 4.4 Assembly & Export of Complete Feature Tables
We assemble the complete feature tables combining:
- **Core Group ($N=41$):** Scope, calendar, network, content rating, genres.
- **Talent Reach Group ($N=6$):** Prior reach max and mean for director, writer, cast.
- **Talent Experience Group ($N=8$, OFF by default):** All-time and 5-year $n_{	ext{prior}}$ counts.
- **Text Group ($N=1$, OFF by default):** `synopsis_length`.
- **Missingness Group ($N=4$, OFF by default):** Ablation indicators.
- **Metadata & Targets ($N=6$):** `name`, `country`, `release_year`, `complete_info`, `target_pct`, `target_tier`.

Saved to gitignored paths:
- `data/processed/train_features_v1.csv` & `data/processed/test_features_v1.csv`
- `data/processed/train_features_v2.csv` & `data/processed/test_features_v2.csv`


In [15]:
meta_cols = ['name', 'country', 'release_year', 'complete_info', 'target_pct', 'target_tier']

# Assemble V1
train_full_v1 = pd.concat([train_df[meta_cols], train_core, train_talent_priors, train_text[['synopsis_length']], train_miss], axis=1)
test_full_v1 = pd.concat([test_df[meta_cols], test_core, test_talent_v1, test_text[['synopsis_length']], test_miss], axis=1)

# Assemble V2
train_full_v2 = pd.concat([train_df[meta_cols], train_core, train_talent_priors, train_text[['synopsis_length']], train_miss], axis=1)
test_full_v2 = pd.concat([test_df[meta_cols], test_core, test_talent_v2, test_text[['synopsis_length']], test_miss], axis=1)

# Save to disk
train_full_v1.to_csv('../data/processed/train_features_v1.csv', index=False)
test_full_v1.to_csv('../data/processed/test_features_v1.csv', index=False)
train_full_v2.to_csv('../data/processed/train_features_v2.csv', index=False)
test_full_v2.to_csv('../data/processed/test_features_v2.csv', index=False)

print("=== EXPORT COMPLETE ===")
print(f"Train Features V1 shape: {train_full_v1.shape} -> saved to data/processed/train_features_v1.csv")
print(f"Test Features V1 shape:  {test_full_v1.shape} -> saved to data/processed/test_features_v1.csv")
print(f"Train Features V2 shape: {train_full_v2.shape} -> saved to data/processed/train_features_v2.csv")
print(f"Test Features V2 shape:  {test_full_v2.shape} -> saved to data/processed/test_features_v2.csv")

assert train_full_v1.columns.tolist() == test_full_v1.columns.tolist(), "Schema mismatch in V1!"
assert train_full_v2.columns.tolist() == test_full_v2.columns.tolist(), "Schema mismatch in V2!"
print("\n[PASSED] Feature tables verified and exported successfully.")


=== EXPORT COMPLETE ===
Train Features V1 shape: (5010, 66) -> saved to data/processed/train_features_v1.csv
Test Features V1 shape:  (2525, 66) -> saved to data/processed/test_features_v1.csv
Train Features V2 shape: (5010, 66) -> saved to data/processed/train_features_v2.csv
Test Features V2 shape:  (2525, 66) -> saved to data/processed/test_features_v2.csv

[PASSED] Feature tables verified and exported successfully.


In [16]:
from src.features import FEATURE_GROUPS, DEFAULT_FEATURE_GROUPS, get_feature_columns

print("=== SWITCHABLE FEATURE GROUPS IN SRC.FEATURES ===")
for g_name, col_list in FEATURE_GROUPS.items():
    status = "ACTIVE (Default)" if g_name in DEFAULT_FEATURE_GROUPS else "OFF by default (Ablation only)"
    print(f"{g_name:18s}: {len(col_list):2d} columns [{status}]")

print(f"\nDefault active features:   {len(get_feature_columns()):2d} columns (core + talent_reach)")
print(f"With text group:           {len(get_feature_columns(include_text=True)):2d} columns")
print(f"With talent_experience:    {len(get_feature_columns(include_experience=True)):2d} columns")
print(f"With all ablation groups:  {len(get_feature_columns(include_experience=True, include_text=True, include_missingness=True)):2d} columns")


=== SWITCHABLE FEATURE GROUPS IN SRC.FEATURES ===
core              : 41 columns [ACTIVE (Default)]
talent_reach      :  6 columns [ACTIVE (Default)]
talent_experience :  8 columns [OFF by default (Ablation only)]
text              :  1 columns [OFF by default (Ablation only)]
missingness       :  4 columns [OFF by default (Ablation only)]

Default active features:   47 columns (core + talent_reach)
With text group:           48 columns
With talent_experience:    55 columns
With all ablation groups:  60 columns
